<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_OB_WEEK_03_1-review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **수상작 리뷰**

제주 특산물 가격 예측 AI 경진대회

https://dacon.io/competitions/official/236176/codeshare

## **주제**
제주도 특산물의 가격을 예측하는 AI 모델 개발 및 인사이트 발굴



## **데이터**

1. train.csv
train 데이터 : 2019년 01월 01일부터 2023년 03월 03일까지의 유통된 품목의 가격 데이터
- item: 품목 코드
  - TG : 감귤
  - BC : 브로콜리
  - RD : 무
  - CR : 당근
  - CB : 양배추
- corporation : 유통 법인 코드
  - 법인 A부터 F 존재
- location : 지역 코드
  - J : 제주도 제주시
  - S : 제주도 서귀포시
- supply(kg) : 유통된 물량, kg 단위
- price(원/kg) : 유통된 품목들의 kg 마다의 가격, 원 단위

2. international_trade.csv
- 관련 품목 수출입 정보
  - 중량 단위 kg
  - 금액 단위 천 달러


3. test.csv
- test 데이터 : 2023년 03월 04일부터 2023년 03월 31일까지의 데이터

## **코드 흐름**

### 1. 데이터 로드 & 시계열 피처 생성
- timestamp 기반 년/월/일/요일/주차 변수 추출

- 연도별 누적 주차(week_num) 계산 및 연말연시 예외 일자 보정

- holidays.KR()을 활용한 공휴일 여부(holiday) 피처 생성

### 2. 품목별 이원화 및 전처리

- 극 이상치 처리: 품목별 상한선(TG>20,000, RD>5,000 등) 초과 값을 nonzero 평균값으로 대체

- 전략 이원화: 가격 변동성이 큰 감귤(TG)과 비감귤(BC, CB, CR, RD) 데이터 분리

- TG 특화 전처리: 공휴일 중 실제 거래가 발생한 날짜의 공휴일 태그 보정, 타겟(price)에 제곱근(np.sqrt) 변환 적용

### 3. 모델링 & 앙상블
- 비감귤류: CatBoost + XGBoost 기반 VotingRegressor 학습

- TG (1/2차): VotingRegressor 및 파라미터를 다르게 한 CatBoostRegressor 모델을 각각 학습 후 예측값 평균 산출

- 복원: 예측값의 음수를 0 처리 후 제곱(np.power) 복원

### 4. 도메인 맞춤형 후처리
- 품목별 3월 거래 최저가 기준 Threshold 미만 예측값을 0원으로 일괄 처리


**주요 코드**

In [ ]:
# [TG 타겟 제곱근 변환 및 CatBoost + XGBoost 앙상블 모델링]
Xy["price"] = np.sqrt(Xy["price"])

cat = CatBoostRegressor(random_state=2024, n_estimators=1000, learning_rate=0.01, depth=10, l2_leaf_reg=3, metric_period=1000)
xgb = XGBRegressor(n_estimators=1000, random_state=2024, learning_rate=0.01, max_depth=10)

vote_model = VotingRegressor(estimators=[("cat", cat), ("xgb", xgb)])
vote_model.fit(Xy.drop(columns=["timestamp", "ID", "price"]), Xy["price"])

pred = vote_model.predict(answer_tg1.drop(columns=["ID"]))
pred = np.where(pred < 0, 0, pred)
answer_tg1["answer"] = np.power(pred, 2)

## **새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

- 농산물 가격처럼 치우침(Skewness)이 심한 타겟 변수에 np.sqrt 변환을 적용하고 예측 후 다시 제곱 복원하는 기법이 모델 안정성 향상에 효과적임을 확인함

- 주 수확/거래 특성이 확연히 다른 특정 품목(TG)을 별도로 분리하여 독립된 모델링 파이프라인을 구축하는 타겟팅 전략을 배움

- ISO calendar 기반 주차 산출 시 12월 말일이 이듬해 1주차로 포함되는 예외 상황 보정 및 52~53주 단위 연도별 누적 주차 계산 로직의 정교한 설계가 까다로웠음

- 단순 모델 성능 향상 외에도, 품목별 최소 거래가 이하의 예측값을 0으로 억제하는 도메인 기반 후처리가 최종 성능(Private 1st) 달성에 결정적 역할을 함을 체득함